In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

from Library.config import BASE_PARAMS
from Library.ODE_timeseries import optimize_patient_parameters

def generate_patient_atlas():
    base_params = BASE_PARAMS.copy()
    base_params['gamma'] = 7.5e-07

    print("1. Loading datasets...")
    params_df = pd.read_csv("IntermediateResults/venex_weighted_params.csv")
    traj_path = Path("InputData/treatment_trajectories.xlsx")
    sheet_dict = pd.read_excel(traj_path, sheet_name=None)
    
    blood_all, bm_all, drug_all = None, None, None
    for sheet_name, df in sheet_dict.items():
        if 'days_lab' in df.columns or 'b_neut' in df.columns:
            blood_all = df
        elif 'days_from_bm' in df.columns or 'bm_blasts' in df.columns:
            bm_all = df
        elif 'venetoclax_start' in df.columns or 'aza_start' in df.columns:
            drug_all = df

    if blood_all is None or bm_all is None or drug_all is None:
        raise ValueError("Missing one or more required sheets (blood, bm, or drug) in treatment_trajectories.xlsx")

    def standardize_id(df):
        col = 'Pseudonym' if 'Pseudonym' in df.columns else ('Patient_ID' if 'Patient_ID' in df.columns else 'ID')
        df['Patient_ID'] = df[col].astype(str).str.strip()
        return df

    blood_all = standardize_id(blood_all)
    bm_all = standardize_id(bm_all)
    drug_all = standardize_id(drug_all)
    params_df['Patient_ID'] = params_df['ID'].astype(str).str.strip()

    patient_ids = params_df['Patient_ID'].unique()
    optimization_jobs = []

    print(f"2. Pre-processing trajectories for {len(patient_ids)} patients...")
    for pid in patient_ids:
        p_row = params_df[params_df['Patient_ID'] == pid].iloc[0]
        phsc = float(p_row.get('pHSC_weighted', 0.95))
        coef = float(p_row.get('coef_weighted', 1.0))

        blood_df = blood_all[blood_all['Patient_ID'] == pid].sort_values('days_lab').copy()
        bm_df = bm_all[bm_all['Patient_ID'] == pid].sort_values('days_from_bm').copy()
        drug_df = drug_all[drug_all['Patient_ID'] == pid].copy()

        if blood_df.empty or bm_df.empty:
            continue

        blood_df.loc[blood_df['days_lab'] < 0, 'days_lab'] = 0
        bm_df.loc[bm_df['days_from_bm'] < 0, 'days_from_bm'] = 0
        blood_df = blood_df.groupby('days_lab', as_index=False)['b_neut'].mean().sort_values('days_lab')
        bm_df = bm_df.groupby('days_from_bm', as_index=False)['bm_blasts'].mean().sort_values('days_from_bm')
        blood_df.loc[blood_df['b_neut'] <= 0, 'b_neut'] = 1e-9
        blood_df['b_neut'] = blood_df['b_neut'] * 5 * 10**9 / 80

        # Unified cycle window: treats the entire active cycle as a single ON block
        schedule = []
        for _, d_row in drug_df.iterrows():
            v_start, v_stop = d_row.get('venetoclax_start', np.nan), d_row.get('venetoclax_stop', np.nan)
            a_start, a_stop = d_row.get('aza_start', np.nan), d_row.get('aza_stop', np.nan)
            cycle_starts, cycle_stops = [], []

            if pd.notna(v_start) and pd.notna(v_stop) and (v_stop >= v_start) and (v_stop >= 0):
                cycle_starts.append(float(v_start))
                cycle_stops.append(float(v_stop))
            if pd.notna(a_start) and pd.notna(a_stop) and (a_stop >= a_start) and (a_stop >= 0):
                cycle_starts.append(float(a_start))
                cycle_stops.append(float(a_stop))
            if cycle_starts and cycle_stops:
                schedule.append((min(cycle_starts), max(cycle_stops)))

        optimization_jobs.append((pid, phsc, coef, base_params.copy(), blood_df, bm_df, schedule))

    print(f"3. Launching mathematical optimizer for {len(optimization_jobs)} patients...")
    results = Parallel(n_jobs=-1)(
        delayed(optimize_patient_parameters)(
            (job[1], job[2], job[3], job[4], job[5], job[6])
        )
        for job in optimization_jobs
    )

    atlas_records = []
    for job, res in zip(optimization_jobs, results):
        pid = job[0]
        if res is not None:
            atlas_records.append({
                "Patient_ID": pid,
                "Predefined_pHSC": res["Predefined_pHSC"],
                "Predefined_Coef": res["Predefined_Coef"],
                "Fitted_DrugKill_LSC": res["Fitted_DrugKill_LSC"],
                "Fitted_DrugKill_HSC": res["Fitted_DrugKill_HSC"],
                "Convergence_Success": res["Convergence_Success"],     
                "Convergence_Message": res["Convergence_Message"],     
                "Combined_Cost": res["RangeScaled_Combined_Error"]
            })
        else:
            print(f"   -> Optimizer failed to converge for patient {pid}")

    atlas_df = pd.DataFrame(atlas_records)
    Path("IntermediateResults").mkdir(parents=True, exist_ok=True)
    atlas_df.to_csv("IntermediateResults/Patient_Atlas.csv", index=False)
    
    print(f"\nFinished! Successfully optimized {len(atlas_df)}/{len(optimization_jobs)} patients.")
    print("Saved to 'IntermediateResults/Patient_Atlas.csv'.")
    return atlas_df

if __name__ == '__main__':
    atlas_df = generate_patient_atlas()